# v3 research run: ranking ensemble on a 500-stock NSE universe

Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.

Stages: build the dataset, train XGBoost (regression + ranker), a GRU on raw candles and a CNN on candlestick chart images, then report walk-forward results after costs. Every test year is scored by models that only saw earlier data.

In [ ]:
import torch, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')
!rm -rf /content/ml-trading-bot /content/eod2_data
!git clone -q --depth 1 https://github.com/arjitmenon9/ml-trading-bot /content/ml-trading-bot
!git clone -q --depth 1 https://github.com/BennyThadikaran/eod2_data /content/eod2_data
!ls /content/eod2_data/daily | wc -l
%cd /content/ml-trading-bot

In [ ]:
!python src/ml/v3/pipeline.py --stage data --eod2 /content/eod2_data --out /content/v3_out

In [ ]:
!python src/ml/v3/pipeline.py --stage trees --out /content/v3_out

In [ ]:
!python src/ml/v3/pipeline.py --stage gru --out /content/v3_out

In [ ]:
!python src/ml/v3/pipeline.py --stage cnn --out /content/v3_out

In [ ]:
!python src/ml/v3/pipeline.py --stage report --out /content/v3_out
from IPython.display import Image, display
display(Image('/content/v3_out/equity.png'))

In [ ]:
!cd /content/v3_out && zip -q -r /content/v3_results.zip report.json trials.jsonl equity.png pred_*.parquet && ls -lh /content/v3_results.zip
print(open('/content/v3_out/report.json').read())